# LLAMA-INDEX BASICS

In [1]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [2]:
# !pip install llama-index-readers-wikipedia wikipedia

In [3]:
from llama_index.readers.wikipedia import WikipediaReader

loader = WikipediaReader()
documents = loader.load_data(pages=['Natural Language Processing', 'Artificial Intelligence'])
print(len(documents))

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): en.wikipedia.org:80
Starting new HTTP connection (1): en.wikipedia.org:80
DEBUG:urllib3.connectionpool:http://en.wikipedia.org:80 "GET /w/api.php?list=search&srprop=&srlimit=1&limit=1&srsearch=Natural+Language+Processing&srinfo=suggestion&format=json&action=query HTTP/1.1" 301 0
http://en.wikipedia.org:80 "GET /w/api.php?list=search&srprop=&srlimit=1&limit=1&srsearch=Natural+Language+Processing&srinfo=suggestion&format=json&action=query HTTP/1.1" 301 0
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): en.wikipedia.org:443
Starting new HTTPS connection (1): en.wikipedia.org:443
DEBUG:urllib3.connectionpool:https://en.wikipedia.org:443 "GET /w/api.php?list=search&srprop=&srlimit=1&limit=1&srsearch=Natural+Language+Processing&srinfo=suggestion&format=json&action=query HTTP/1.1" 200 174
https://en.wikipedia.org:443 "GET /w/api.php?list=search&srprop=&srlimit=1&limit=1&srsearch=Natural+Language+Processing&srinfo=su

In [4]:
documents[1].text

'Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI 

In [5]:
from llama_index.core.node_parser import SimpleNodeParser

# Assuming documents have already been loaded

# Initialize the parser
parser = SimpleNodeParser.from_defaults(chunk_size=512, chunk_overlap=20)

# Parse documents into nodes
nodes = parser.get_nodes_from_documents(documents)
print(len(nodes))

DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: Natural language processing (NLP) is the proces...
> Adding chunk: Natural language processing (NLP) is the proces...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: When the "patient" exceeded the very small know...
> Adding chunk: When the "patient" exceeded the very small know...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 1990s: Many of the notable early successes in s...
> Adding chunk: 1990s: Many of the notable early successes in s...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: This is increasingly important in medicine and ...
> Adding chunk: This is increasingly important in medicine and ...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: === Neural networks ===

A major drawback of st...
> Adding chunk: === Neural networks ===

A major drawback of st...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: === Text and speech processing ===

Op

# Indexing

In [6]:
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [7]:
import pinecone
print(pinecone.__file__)

e:\anaconda3\envs\MY_RAG\Lib\site-packages\pinecone\__init__.py


In [9]:
import os
from pinecone import Pinecone, ServerlessSpec
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

from dotenv import load_dotenv

load_dotenv()

# Instantiate the Pinecone client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "instruction-manual-rag-langchain"

# Create the index if it doesn't already exist
if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,  # text-embedding-3-small dimension
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Connect to the index
pinecone_index = pc.Index(index_name)

# Wrap it as a LlamaIndex vector store
vector_store = PineconeVectorStore(
    pinecone_index=pinecone_index,
    namespace="tenant-123",
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [10]:
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
)

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-a30f9f93-9585-414d-a784-429a0738c5bb', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000002BE112ECEA0>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly. Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.   == History ==  Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence," which propos

Upserted vectors:   0%|          | 0/58 [00:00<?, ?it/s]

# Query Engine

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

# Optional but recommended: be explicit about which LLM does the answering
Settings.llm = OpenAI(model="gpt-4o-mini")  # or "gpt-4o", "gpt-3.5-turbo", etc.

# Build the query engine
query_engine = index.as_query_engine(
    similarity_top_k=5,       # how many chunks to retrieve before synthesizing
    response_mode="compact",  # how retrieved chunks get combined for the LLM
)

# Ask a question
response = query_engine.query("what is NLP")
print(response)

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-ea3ece1b-5e3b-4e70-a10f-8eb9a4ae53fb', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000002BE12E51440>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['what is NLP'], 'model': 'text-embedding-3-small', 'encoding_format': 'base64'}}
Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-ea3ece1b-5e3b-4e70-a10f-8eb9a4ae53fb', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000002BE12E51440>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['what is NLP'], 'model': 'text-embedding-3-small', 'encoding_format': 'base64'}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://api.openai.com/v1/embeddings
Sending HTTP Request: POST https://api.openai.com/v1/embeddings
DEBUG:httpcore.connection:clos

In [15]:
response = query_engine.query("what is NLP")
print(response)

for node in response.source_nodes:
    print(node.score, node.text[:200])

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-078a928c-a4a8-4917-a722-7704cf2f09a1', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000002BE05F28F40>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['what is NLP'], 'model': 'text-embedding-3-small', 'encoding_format': 'base64'}}
Request options: {'method': 'post', 'url': '/embeddings', 'files': None, 'idempotency_key': 'stainless-python-retry-078a928c-a4a8-4917-a722-7704cf2f09a1', 'post_parser': <function Embeddings.create.<locals>.parser at 0x000002BE05F28F40>, 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'input': ['what is NLP'], 'model': 'text-embedding-3-small', 'encoding_format': 'base64'}}
DEBUG:openai._base_client:Sending HTTP Request: POST https://api.openai.com/v1/embeddings
Sending HTTP Request: POST https://api.openai.com/v1/embeddings
DEBUG:httpcore.connection:clos

In [16]:
for node in response.source_nodes:
    print(node.score, node.text[:200])

0.6115588695966127 Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is 
0.5830970254335687 === Cognition ===
Most higher-level NLP applications involve aspects that emulate intelligent behavior and apparent comprehension of natural language. More broadly speaking, the technical operationali


# Routing

In [17]:
from llama_index.core import SummaryIndex

# Your existing Pinecone-backed vector engine
vector_query_engine = index.as_query_engine()

# A second engine for whole-document summarization
summary_index = SummaryIndex(nodes)
list_query_engine = summary_index.as_query_engine()

In [18]:
from llama_index.core.tools import QueryEngineTool

list_tool = QueryEngineTool.from_defaults(
    query_engine=list_query_engine,
    description="Useful for summarization questions related to the instruction manual",
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific context related to the instruction manual",
)

In [19]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import PydanticSingleSelector

query_engine = RouterQueryEngine(
    selector=PydanticSingleSelector.from_defaults(),
    query_engine_tools=[
        list_tool,
        vector_tool,
    ],
)

In [20]:
response = query_engine.query("what things do we learn in Artificial Intelligence")
print(response)

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-2089eb38-b3ec-4d55-b02f-6ed6859b2773', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'role': 'user', 'content': "Some choices are given below. It is provided in a numbered list (1 to 2), where each item in the list corresponds to a summary.\n---------------------\n(1) Useful for summarization questions related to the instruction manual\n\n(2) Useful for retrieving specific context related to the instruction manual\n---------------------\nUsing only the choices above and not prior knowledge, generate the selection object and reason that is most relevant to the question: 'what things do we learn in Artificial Intelligence'\n"}], 'model': 'gpt-4o-mini', 'parallel_tool_calls': False, 'stream': False, 'temperature': 0.1, 'tool_choice': 'required', 'tools': [{'type': 'function', 'function': {'name': 'SingleSelect

In [21]:
response

Response(response='In the field of Artificial Intelligence, we learn about various capabilities and techniques that enable machines to perform tasks typically associated with human intelligence. Key areas of focus include:\n\n1. **Reasoning and Problem-Solving**: Understanding algorithms that mimic human reasoning, including methods for dealing with uncertain information and making logical deductions.\n\n2. **Knowledge Representation**: Learning how to represent information about the world in a form that AI systems can utilize for reasoning and decision-making.\n\n3. **Planning and Decision-Making**: Exploring how agents can set goals and make decisions based on preferences and expected outcomes.\n\n4. **Learning**: Studying different types of machine learning, including supervised, unsupervised, and reinforcement learning, as well as deep learning techniques.\n\n5. **Natural Language Processing (NLP)**: Gaining insights into how machines can understand, interpret, and generate human l